# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/workspace/default/staynest_07"
CATALOG = "staynest07_catalog"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
from pyspark.sql.functions import broadcast

# hotels also has a city column, so hotels_df.drop("city") before joining
hotels_slim = hotels_df.drop("city")   # bookings already carries city; avoids a duplicate column

# --- Plan 1:Join bookings to hotels and call .explain() to see the plan ---
plain_join = bookings_df.join(hotels_slim, "hotel_id")
plain_join.explain()

# --- Plan 2: forced broadcast of the small table ---
broadcast_join = bookings_df.join(broadcast(hotels_slim), "hotel_id")
broadcast_join.explain()

# Plan 1 used: <fill in after running: BroadcastHashJoin or SortMergeJoin>.
# Plan 2 used: BroadcastHashJoin. Hotels (200 rows) is copied to every executor
# A broadcast join needs no shuffle because every task already holds the full small
# table and joins its own bookings partition locally; a SortMergeJoin would have to
# shuffle and sort both sides by hotel_id first.


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [hotel_id#11302, booking_id#11300, customer_id#11301, booking_date#11303, city#11304, nights#11305, amount#11306, status#11307, hotel_name#11359, category#11361, star_rating#11362]
         +- PhotonBroadcastHashJoin [hotel_id#11302], [hotel_id#11358], Inner, BuildRight, false, true, false
            :- PhotonFilter isnotnull(hotel_id#11302)
            :  +- PhotonRowToColumnar
            :     +- FileScan csv [booking_id#11300,customer_id#11301,hotel_id#11302,booking_date#11303,city#11304,nights#11305,amount#11306,status#11307] Batched: false, DataFilters: [isnotnull(hotel_id#11302)], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/staynest_07/bookings.csv], PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking_date:date,city:s

## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:
from pyspark.sql import functions as F

target = FQN("bookings_delta")   # workspace.default.bookings_delta

# Check the real status values first (spelling/case must match the WHERE clauses)
bookings_df.groupBy("status").count().show()

# --- Version 0: WRITE ---
spark.sql(f"DROP TABLE IF EXISTS {target}")   # makes the cell safe to re-run from a clean history
(bookings_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")   # allows re-runs even after schema evolution
    .format("delta")
    .saveAsTable(target))

# --- Version 1: UPDATE ---
spark.sql(f"UPDATE {target} SET status = 'completed' WHERE status = 'pending'")

# --- Version 2: DELETE ---
spark.sql(f"DELETE FROM {target} WHERE status = 'cancelled'")

# --- Read the history ---
display(spark.sql(f"DESCRIBE HISTORY {target}")
        .select("version", "timestamp", "operation"))



+---------+-----+
|   status|count|
+---------+-----+
|  pending|  903|
|cancelled| 1437|
|completed| 9660|
+---------+-----+



version,timestamp,operation
2,2026-09-25T13:56:14.000Z,DELETE
1,2026-09-25T13:56:11.000Z,UPDATE
0,2026-09-25T13:56:03.000Z,CREATE OR REPLACE TABLE AS SELECT


## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
target = FQN("bookings_delta")

# Current count (after UPDATE + DELETE)
current_count = spark.table(target).count()

# Read version 0 with the DataFrame reader
v0_df = (spark.read
         .format("delta")
         .option("versionAsOf", 0)
         .table(target))

v0_count = v0_df.count()
print(f"Current: {current_count} | Version 0: {v0_count}")   # expect these to differ

# Roll back
spark.sql(f"RESTORE TABLE {target} TO VERSION AS OF 0")

# Confirm the count is back
restored_count = spark.table(target).count()
print("After RESTORE:", restored_count)                      # should equal v0_count

# History again: look for the new RESTORE row on top
display(spark.sql(f"DESCRIBE HISTORY {target}")
        .select("version", "timestamp", "operation")
        .orderBy("version", ascending=False))


Current: 10563 | Version 0: 12000
After RESTORE: 12000


version,timestamp,operation
4,2026-09-25T13:56:43.000Z,RESTORE
3,2026-09-25T13:56:18.000Z,OPTIMIZE
2,2026-09-25T13:56:14.000Z,DELETE
1,2026-09-25T13:56:11.000Z,UPDATE
0,2026-09-25T13:56:03.000Z,CREATE OR REPLACE TABLE AS SELECT


## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:
target = FQN("bookings_delta")

# File layout before
display(spark.sql(f"DESCRIBE DETAIL {target}").select("numFiles", "sizeInBytes"))

# 1. Compact small files
display(spark.sql(f"OPTIMIZE {target}"))

# 2. Compact and colocate rows by city
display(spark.sql(f"OPTIMIZE {target} ZORDER BY (city)"))

# OPTIMIZE rewrites many small files into fewer, larger ones (bin-packing), which cuts
# per-file overhead when reading. It is a new commit; the old files stay for time travel.

# city is a sensible ZORDER column because it has many distinct values and is likely
# to be used in filters (WHERE city = ...). ZORDER sorts rows by city so each file
# covers a narrow range of cities, and data skipping can then skip files whose
# min/max city stats can't match the filter.
# status would not help: with only three values, nearly every file contains every
# status, so min/max stats can't rule out any file and there is nothing to skip.


numFiles,sizeInBytes
1,113161


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1790344616261, 1790344618703, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null, 0)"


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 113161), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1790344620065, 1790344623037, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null, 0)"


## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# ── Ingest raw CSV → Bronze Delta with metadata ──
bronze_bookings = (spark.read.csv(f"{BASE}/bookings.csv",
                                header=True, inferSchema=True)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("bookings.csv"))
)

(bronze_bookings.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")   # safe to re-run
    .saveAsTable(FQN("bronze_bookings")))

print(f"✅ Bronze table created: {FQN('bronze_bookings')}")
spark.table(FQN("bronze_bookings")).select(
    "booking_id", "amount", "_ingested_at", "_source_file"
).show(3, truncate=False)


✅ Bronze table created: staynest07_catalog.default.bronze_bookings
+----------+-------+--------------------------+------------+
|booking_id|amount |_ingested_at              |_source_file|
+----------+-------+--------------------------+------------+
|9000000   |6087.65|2026-09-25 13:57:11.866706|bookings.csv|
|9000001   |8211.19|2026-09-25 13:57:11.866706|bookings.csv|
|9000002   |7176.52|2026-09-25 13:57:11.866706|bookings.csv|
+----------+-------+--------------------------+------------+
only showing top 3 rows


## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
from pyspark.sql.functions import col

# Check the hotel-side column names (category, star_rating and the hotel name column)
print(hotels_df.columns)

silver_df = (spark.table(FQN("bronze_bookings"))
    .filter(col("status") == "completed")
    .join(hotels_df.drop("city"), "hotel_id"))    # bookings keeps its city; hotels' copy is dropped

(silver_df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(FQN("silver_bookings")))

# Verify
s = spark.table(FQN("silver_bookings"))
print("silver rows:", s.count())
print("city columns:", s.columns.count("city"))        # expect 1
print("statuses:", [r[0] for r in s.select("status").distinct().collect()])   # expect ['completed']
assert s.columns.count("city") == 1
# assert s.filter(col("status") != "completed").count() == 0
display(s.limit(5))


['hotel_id', 'hotel_name', 'city', 'category', 'star_rating']
silver rows: 9660
city columns: 1
statuses: ['completed']


hotel_id,booking_id,customer_id,booking_date,city,nights,amount,status,_ingested_at,_source_file,hotel_name,category,star_rating
3095,9000000,701600,2025-11-27,Jaipur,4,6087.65,completed,2026-09-25T13:57:11.866Z,bookings.csv,Orchid Suites,Budget,3.8
3112,9000003,700867,2025-03-22,Bengaluru,5,7880.62,completed,2026-09-25T13:57:11.866Z,bookings.csv,Grand Stay,Budget,3.6
3012,9000006,701336,2025-11-25,Delhi,7,70999.15,completed,2026-09-25T13:57:11.866Z,bookings.csv,Orchid Residency,Luxury,4.1
3127,9000007,700868,2025-04-10,Mumbai,2,15693.64,completed,2026-09-25T13:57:11.866Z,bookings.csv,Summit Stay,Luxury,4.5
3045,9000008,700687,2025-01-15,Mumbai,1,1492.62,completed,2026-09-25T13:57:11.866Z,bookings.csv,Sunset Inn,Budget,3.0


## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
from pyspark.sql.functions import count, sum as _sum, col, desc

SILVER = FQN("silver_bookings")
GOLD   = FQN("gold_city_revenue")

gold_df = (spark.table(SILVER)
    .groupBy("city")
    .agg(count("*").alias("bookings"),
         _sum("amount").alias("revenue"))
    .orderBy(desc("revenue")))

(gold_df.write
    .mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD))

# Verify
g = spark.table(GOLD)
display(g.orderBy(desc("revenue")))    # re-sort on read; see note below
print("cities:", g.count())

# Totals should reconcile with silver
silver_total = spark.table(SILVER).agg(_sum("amount")).first()[0]
gold_total = g.agg(_sum("revenue")).first()[0]
print("silver revenue:", silver_total, "| gold revenue:", gold_total)


city,bookings,revenue
Goa,2546,4.459670179E7
Mumbai,1715,3.624122111999999E7
Delhi,1174,2.631428154000002E7
Jaipur,979,2.4436853129999984E7
Bengaluru,1318,2.2670136969999984E7
Udaipur,691,1.2094427419999994E7
Rishikesh,407,8606121.57999999
Manali,480,6235480.679999996
Munnar,244,3979216.11
Anantapur,106,2257080.65


cities: 10
silver revenue: 187431520.99000034 | gold revenue: 187431520.98999998


## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
# Roll back to the pre-merge state (version 0, the original load)
spark.sql(f"RESTORE TABLE {FQN("bookings_delta")} TO VERSION AS OF 0")
print("Reset to:", spark.table(FQN("bookings_delta")).count())   # expect 12000
# Sanity checks before merging
print("bookings_delta cols match batch:", set(spark.table(FQN("bookings_delta")).columns) == set(updates_df.columns))
print("batch rows:", updates_df.count(),
      "| distinct booking_ids:", updates_df.select("booking_id").distinct().count())   # expect 200 / 200

updates_df.createOrReplaceTempView("batch")

# Count BEFORE
before = spark.table(FQN("bookings_delta")).count()
print("Before MERGE:", before)        # expect 12000

# One MERGE: update matched, insert new
spark.sql(f"""
    MERGE INTO {FQN("bookings_delta")} AS t
    USING batch AS s
    ON t.booking_id = s.booking_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Count AFTER
after = spark.table(FQN("bookings_delta")).count()
print("After MERGE:", after)          # expect 12050
print("Net new rows:", after - before)
assert after - before == 50

# Prove the split: 150 updated, 50 inserted
m = (spark.sql(f"DESCRIBE HISTORY {FQN("bookings_delta")} LIMIT 1")
        .select("version", "operation", "operationMetrics").first())
print(m["operation"], {k: v for k, v in m["operationMetrics"].items()
                       if k in ("numTargetRowsUpdated", "numTargetRowsInserted", "numTargetRowsDeleted")})


Reset to: 12000
bookings_delta cols match batch: True
batch rows: 200 | distinct booking_ids: 200
Before MERGE: 12000
After MERGE: 12050
Net new rows: 50
MERGE {'numTargetRowsDeleted': '0', 'numTargetRowsUpdated': '150', 'numTargetRowsInserted': '50'}
